In [2]:
!pip install kagglehub

In [3]:
import kagglehub
path = kagglehub.dataset_download("smayanj/netflix-users-database")
print("Path to dataset files:", path)

100%|██████████| 354k/354k [00:00<00:00, 48.8MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/smayanj/netflix-users-database/versions/1


# **Датасет**
В качестве датасета я выбрал Netflix Userbase Dataset, в котором представлены различные данные об активности пользователей Netflix.

In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from itertools import combinations
import os

file_name = os.listdir(path)[0]
df = pd.read_csv(os.path.join(path, file_name))

print(df.head())
print(df.info())

   User_ID            Name  Age Country Subscription_Type  Watch_Time_Hours  \
0        1  James Martinez   18  France           Premium             80.26   
1        2     John Miller   23     USA           Premium            321.75   
2        3      Emma Davis   60      UK             Basic             35.89   
3        4     Emma Miller   44     USA           Premium            261.56   
4        5      Jane Smith   68     USA          Standard            909.30   

  Favorite_Genre  Last_Login  
0          Drama  2024-05-12  
1         Sci-Fi  2025-02-05  
2         Comedy  2025-01-24  
3    Documentary  2024-03-25  
4          Drama  2025-01-14  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   User_ID            25000 non-null  int64  
 1   Name               25000 non-null  object 
 2   Age                25000 non-null 

# **Двухвыборочные гипотезы касательно медиан и распределений для непрерывного случая**
Снова будем сравнивать две группы пользователей с типами подписки Basic и Premium. В качестве непрерывной величины возьмём время просмотра (Watch_Time_Hours). Сформулируем гипотезы о равенстве медиан и распределений.

Так как величины непрерывные, мы можем использовать критерий Колмогорова-Смирнова для сравнения распределений двух функций. P-value значительно больше 0.05, а значит распределения примерно одинаковые. В таком случае, для проверки равенства медиан распределений мы можем уверенно использовать критерий Манна-Уитни. Вновь получаем большой P-value, а значит и медианы у двух распределений примерно равны.



In [5]:
group_p = df[df['Subscription_Type'] == 'Premium']
group_b = df[df['Subscription_Type'] == 'Basic']

results = []

for metric_name, metric_data in [('Watch_Time_Hours', 'непрерывный')]:
    data_p = group_p[metric_name].dropna()
    data_b = group_b[metric_name].dropna()

    u_stat, p_median = stats.mannwhitneyu(data_p, data_b)

    ks_stat, p_dist = stats.ks_2samp(data_p, data_b)

    results.append({
        "Метрика": metric_name,
        "Тип": metric_data,
        "P-value (Медианы)": round(p_median, 4),
        "P-value (Распределения)": round(p_dist, 4)
    })

df_res = pd.DataFrame(results)
display(df_res)

,Метрика,Тип,P-value (Медианы),P-value (Распределения)
0,Watch_Time_Hours,непрерывный,0.7146,0.8752


# **Двухвыборочные гипотезы касательно медиан и распределений для дискретного случая**
В качестве дискретной величины возьмём возраст (Age). С дискретным случаем всё несколько сложнее, мы не можем уверенно использовать критерий Колмогорова-Смирнова для проверки равенства распределений. Вместо него используем Критерий Хи-квадрат. Поскольку возраст дискретен, Хи-квадрат лучше классического Колмогорова, так как он работает с частотами конкретных значений, а не требует непрерывности.
Для проверки равенства медиан используем медианный критерий Муда, так как он менее чувствителен к большому количеству повторяющихся значений, которые могут искажать ранговую статистику критерия Манна-Уитни.
Как мы видим, P-value в обоих случаях значительно превышает порог 0.05, а значит обе нулевые гипотезы подтверждаются.

In [8]:
age_premium = df[df['Subscription_Type'] == 'Premium']['Age']
age_basic = df[df['Subscription_Type'] == 'Basic']['Age']

stat_mood, p_mood, grand_median, contingency_mood = stats.median_test(age_premium, age_basic)

contingency_table = pd.crosstab(df['Age'], df['Subscription_Type'])
chi2, p_chi2, dof, expected = stats.chi2_contingency(contingency_table)

results_discrete = pd.DataFrame({
    "Гипотеза": ["Равенство медиан", "Идентичность распределений"],
    "p-value": [round(p_mood, 4), round(p_chi2, 4)]
})

display(results_discrete)

,Гипотеза,p-value
0,Равенство медиан,0.4578
1,Идентичность распределений,0.7485


# **Проверка бутстрапом**

Доверительные интервалы, полученные методом Эфрона, полностью подтверждают результаты классических тестов. Оба интервала разницы между медианами распределений включают 0. Это означает, что на уровне значимости 5% мы не можем отвергнуть гипотезу о равенстве медиан — статистически значимых различий нет.
95% ДИ расстояния между распределениями для обоих показателей крайне малы (диапазон 0.009 – 0.027), что подтверждает идентичность форм распределений в группах Premium и Basic.

In [6]:
def bootstrap_median_diff(s1, s2, n=1000):
    diffs = []
    for _ in range(n):
        # Генерируем выборки с возвращением
        sample1 = np.random.choice(s1, size=len(s1), replace=True)
        sample2 = np.random.choice(s2, size=len(s2), replace=True)
        diffs.append(np.median(sample1) - np.median(sample2))
    return np.percentile(diffs, [2.5, 97.5])

def bootstrap_ks_dist(s1, s2, n=1000):
    ks_values = []
    for _ in range(n):
        sample1 = np.random.choice(s1, size=len(s1), replace=True)
        sample2 = np.random.choice(s2, size=len(s2), replace=True)

        d_stat, _ = stats.ks_2samp(sample1, sample2)
        ks_values.append(d_stat)

    return np.percentile(ks_values, [2.5, 97.5])

ci_median_watch = bootstrap_median_diff(group_p['Watch_Time_Hours'], group_b['Watch_Time_Hours'])
ci_median_age = bootstrap_median_diff(group_p['Age'], group_b['Age'])
dist_ci_watch = bootstrap_ks_dist(group_p['Watch_Time_Hours'], group_b['Watch_Time_Hours'])
dist_ci_age = bootstrap_ks_dist(group_p['Age'], group_b['Age'])

print(f"95% ДИ разницы медиан (Часы): {ci_median_watch}")
print(f"95% ДИ разницы медиан (Возраст): {ci_median_age}")
print(f"95% ДИ расстояния между распределениями (Watch Time): {dist_ci_watch}")
print(f"95% ДИ расстояния между распределениями (Age): {dist_ci_age}")

95% ДИ разницы медиан (Часы): [-13.36175   14.310375]
95% ДИ разницы медиан (Возраст): [-1.  1.]
95% ДИ расстояния между распределениями (Watch Time): [0.01023286 0.02638574]
95% ДИ расстояния между распределениями (Age): [0.00954658 0.02766466]


Самым мощным и надёжным методом, безусловно, является бутстрап. Во-первых, он не требует никаких ограничений на выборки, и это позволяет ему учитывать любые микро-особенности как для непрерывного, так и для дискретного случая. Во-вторых, бутстрап даёт не просто уверенность p-value, а конкретные значения - доверительный интервал. Единственная проблема использования бутстрапа в его ресурсоёмкости и времени работы. В моём случае выборки довольно большие, так что проблем с малым количеством данных не наблюдается.